# AI Code Auditor v2 — Live Demo
**Model:** DeepSeek-Coder-6.7B + QLoRA | **Dataset:** Big-Vul (top-10 CWEs)

### Before running:
1. Set GPU to T4 x1
2. Attach `models/lora-adapter-v2` dataset
3. Run All — takes ~3 min to load
4. Share the public Gradio URL

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 accelerate==0.29.3 bitsandbytes==0.45.3 gradio
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
print('Done')

In [ ]:
import os, torch, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'adapter_config.json' and 'checkpoint' not in root:
            ADAPTER_PATH = root

assert ADAPTER_PATH, 'adapter_config.json not found'
print(f'Adapter: {ADAPTER_PATH}')
print(f'CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}')

BASE_MODEL = 'deepseek-ai/deepseek-coder-6.7b-base'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL,
    quantization_config=bnb, device_map={'': 0},
    trust_remote_code=True, torch_dtype=torch.float16)
base_model.config.use_cache = True
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print(f'Model ready! VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

In [ ]:
import gradio as gr
import torch, re

def analyze_code(code):
    if not code or len(code.strip()) < 10:
        return 'Please paste some code to analyze.', '', ''

    sys = 'You are an expert security code auditor. Analyze C/C++ code for vulnerabilities, classify them using CWE, and rewrite the code securely.'
    prompt = '<s>[INST] <<SYS>>\n' + sys + '\n<</SYS>>\n\nAnalyze the following C/C++ code and identify the security vulnerability.\n\n```c\n' + code.strip() + '\n```\n\nRespond with the CWE type first, then explain and provide a secure rewrite. [/INST] CWE:'

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=450).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=300, do_sample=True,
            temperature=0.2, top_p=0.9, repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
    raw = 'CWE:' + tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    cwe_m  = re.search(r'CWE-\d+', raw)
    cve_m  = re.search(r'CVE:[\s]*(CVE-[\d-]+)', raw)
    cvss_m = re.search(r'Severity:[\s]*([\d.]+\s*\(\w+\))', raw)
    code_b = re.findall(r'```(?:c|cpp)?\n(.*?)```', raw, re.DOTALL)
    reason = re.search(r'Reason:[\s]*(.*?)(?=Fix:|Changes:|$)', raw, re.DOTALL)

    cwe    = cwe_m.group(0)          if cwe_m   else 'Unknown'
    cve    = cve_m.group(1)          if cve_m   else 'N/A'
    cvss   = cvss_m.group(1)         if cvss_m  else 'N/A'
    secure = code_b[-1].strip()      if code_b  else 'See full output below'
    expl   = reason.group(1).strip() if reason  else ''

    emoji = '🔴' if 'HIGH' in cvss.upper() or 'CRITICAL' in cvss.upper() else '🟡'
    summary = emoji + ' **' + cwe + '** detected\n\n📋 **CVE:** ' + cve + '\n⚠️ **CVSS:** ' + cvss + '\n\n**Explanation:**\n' + expl

    return summary, secure, raw

EXAMPLES = [
    ['void copy_input(char *user_input) {\n    char buffer[128];\n    strcpy(buffer, user_input);\n    printf("Input: %s\\n", buffer);\n}'],
    ['int read_data(int fd) {\n    char buf[256];\n    int n = read(fd, buf, 1024);\n    buf[n] = 0;\n    return n;\n}'],
    ['void process(int *arr, int size) {\n    int total = 0;\n    for (int i = 0; i <= size; i++) {\n        total += arr[i];\n    }\n}'],
]

with gr.Blocks(title='AI Code Auditor v2') as demo:
    gr.Markdown('# 🔐 AI Code Auditor v2\n### DeepSeek-Coder-6.7B + QLoRA | Big-Vul Dataset\nPaste C/C++ code and click **Analyze**.')
    with gr.Row():
        with gr.Column(scale=1):
            code_input = gr.Code(label='Paste Vulnerable Code Here', language='c', lines=15)
            analyze_btn = gr.Button('🔍 Analyze Code', variant='primary', size='lg')
            gr.Examples(examples=EXAMPLES, inputs=code_input, label='Example Snippets')
        with gr.Column(scale=1):
            summary_out = gr.Markdown(label='Vulnerability Summary')
            secure_out  = gr.Code(label='Secure Rewrite', language='c', lines=15, interactive=False)
    with gr.Accordion('Full Raw Output', open=False):
        raw_out = gr.Markdown()
    analyze_btn.click(fn=analyze_code, inputs=code_input, outputs=[summary_out, secure_out, raw_out])
    gr.Markdown('---\n**Model:** DeepSeek-Coder-6.7B | **Method:** QLoRA (4-bit NF4, LoRA r=16) | **Dataset:** Big-Vul top-10 CWEs')

demo.launch(share=True)
print('Share the URL above with your faculty!')